## Inference under FHE for the MNIST Dataset using helayers

In this demo, we'll deal with a classification problem for the MNIST dataset [1], trying to correctly classify a batch of samples using a neural network model that will be created and trained using the Keras library (with architecture similar to reference [2]).
First, we'll build a plain neural network for the MNIST model. Then, we'll encrypt the trained network and run inference over it using FHE.

In [1]:
import os

##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)
import random
random.seed(seed_value)
import numpy as np
np.random.seed(seed_value)
import tensorflow as tf
tf.random.set_seed(seed_value)
from tensorflow.keras import backend as K

from tensorflow.keras import utils, losses
import numpy as np
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
import h5py

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)

# batch_size = 16
# batch_size = 32
# batch_size = 64
# batch_size = 128
# batch_size = 256
# batch_size = 512
batch_size = 1024
# batch_size = 2048
# batch_size = 4096
# batch_size=8192
print("Misc. initializations")

2025-12-23 07:27:17.411531: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-23 07:27:17.414427: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/usr/local/cuda-12.2/lib64
2025-12-23 07:27:17.414437: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


Misc. initializations


### Load and Preprocess the MNIST Dataset. 

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

# # 对图像进行裁剪，以适应新的形状 (24, 24, 1)
# x_train = x_train[:, 2:26, 2:26]
# x_test = x_test[:, 2:26, 2:26]

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

x_train /= 255
x_test /= 255
print('data ready')



data ready


In [3]:
# Create validation data
# testSize=16
# testSize=500
testSize=5000
x_val = x_test[:-testSize]
x_test = x_test[-testSize:]
y_val = y_test[:-testSize]
y_test = y_test[-testSize:]
print('Validation and test data ready')

# Convert class vector to binary class matrices
# num_classes = 10
# y_train = utils.to_categorical(y_train, num_classes)
# y_test = utils.to_categorical(y_test, num_classes)
# y_val = utils.to_categorical(y_val, num_classes)

input_shape = x_train[0].shape
print(f'input shape: {input_shape}')

Validation and test data ready
input shape: (28, 28, 1)


### Save Dataset

In [4]:
def save_data_set(x, y, data_type, s=''):
    fname=os.path.join(PATH, f'x_{data_type}{s}.h5')
    print("Saving x_{} of shape {} in {}".format(data_type, x.shape,fname))
    xf = h5py.File(fname, 'w')
    xf.create_dataset('x_{}'.format(data_type), data=x)
    xf.close()

    yf = h5py.File(os.path.join(PATH, f'y_{data_type}{s}.h5'), 'w')
    yf.create_dataset(f'y_{data_type}', data=y)
    yf.close()

save_data_set(x_test, y_test, data_type='test3')
save_data_set(x_train, y_train, data_type='train')
save_data_set(x_val, y_val, data_type='val')

Saving x_test3 of shape (5000, 28, 28, 1) in data/net_mnist/x_test3.h5
Saving x_train of shape (60000, 28, 28, 1) in data/net_mnist/x_train.h5
Saving x_val of shape (5000, 28, 28, 1) in data/net_mnist/x_val.h5


In [5]:
print(f"x_val shape: {x_val.shape}, y_val shape: {y_val.shape}")  # 检查数据量

x_val shape: (5000, 28, 28, 1), y_val shape: (5000,)


### Build a Plain Neural Network for the MNIST Model

In [6]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D, Reshape,AveragePooling2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2
# 创建模型
model = Sequential()

# 保持前四层不变
model.add(Conv2D(20, kernel_size=5, kernel_regularizer=l2(0.0005),input_shape=(28, 28, 1)))
model.add(AvgPool2D(pool_size=2))

model.add(Conv2D(50, kernel_size=5, kernel_regularizer=l2(0.0005)))
model.add(AvgPool2D(pool_size=2))  # 输出 (4,4,50)

# 关键修改：直接输出10通道
model.add(Conv2D(10, kernel_size=(4,4), kernel_regularizer=l2(0.0005)))  # 输出形状 (1,1,10)
model.add(BatchNormalization())
model.add(ReLU())

# 替换全局平均池化为普通平均池化+展平
model.add(AvgPool2D(pool_size=(1,1)))  # 保持空间维度 (1x1)
model.add(Flatten())                   # 输出形状 (10,)
# 输出模型架构


model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 20)        200       
                                                                 
 average_pooling2d (AverageP  (None, 13, 13, 20)       0         
 ooling2D)                                                       
                                                                 
 batch_normalization (BatchN  (None, 13, 13, 20)       80        
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 50)        9050      
                                                                 
 re_lu (ReLU)                (None, 11, 11, 50)        0         
                                                                 
 batch_normalization_1 (Batc  (None, 11, 11, 50)       2

2025-12-23 07:27:18.949275: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: /usr/lib64/libcuda.so.1: file too short; LD_LIBRARY_PATH: :/usr/local/cuda-12.2/lib64
2025-12-23 07:27:18.949296: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2025-12-23 07:27:18.949308: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (627b7895b898): /proc/driver/nvidia/version does not exist
2025-12-23 07:27:18.949744: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


### 第一阶段

In [7]:
#一阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow_addons as tfa

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=10,         # 如果10个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
# def lr_scheduler(epoch):
#     if epoch < 100:
#         return 0.001
#     elif epoch < 200:
#         return 0.0005
#     else:
#         return 0.0001
callbacks = [
    # tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

model.fit(
    x_train, y_train, batch_size=batch_size,
    
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True
              ,
              callbacks=callbacks
              )

score = model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

Epoch 1/4000


/root/miniconda/envs/py310/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/root/miniconda/envs/py310/lib/python3.10/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.9.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If

59/59 - 4s - loss: 2.1415 - accuracy: 0.2657 - val_loss: 2.3167 - val_accuracy: 0.0948 - 4s/epoch - 70ms/step
Epoch 2/4000
59/59 - 4s - loss: 1.9798 - accuracy: 0.3537 - val_loss: 2.3119 - val_accuracy: 0.1368 - 4s/epoch - 63ms/step
Epoch 3/4000
59/59 - 4s - loss: 1.8936 - accuracy: 0.4220 - val_loss: 2.3053 - val_accuracy: 0.1484 - 4s/epoch - 63ms/step
Epoch 4/4000
59/59 - 4s - loss: 1.8333 - accuracy: 0.4737 - val_loss: 2.2816 - val_accuracy: 0.1574 - 4s/epoch - 63ms/step
Epoch 5/4000
59/59 - 4s - loss: 1.7883 - accuracy: 0.5059 - val_loss: 2.2343 - val_accuracy: 0.1672 - 4s/epoch - 63ms/step
Epoch 6/4000
59/59 - 4s - loss: 1.7494 - accuracy: 0.5356 - val_loss: 2.1587 - val_accuracy: 0.2306 - 4s/epoch - 63ms/step
Epoch 7/4000
59/59 - 4s - loss: 1.7116 - accuracy: 0.5568 - val_loss: 2.0631 - val_accuracy: 0.3410 - 4s/epoch - 64ms/step
Epoch 8/4000
59/59 - 4s - loss: 1.6765 - accuracy: 0.5763 - val_loss: 1.9639 - val_accuracy: 0.4266 - 4s/epoch - 63ms/step
Epoch 9/4000
59/59 - 4s - los

In [ ]:
# model.get_weights()

### Rebuild Model

In [8]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2
new_model = Sequential()
for layer in model.layers:
    if isinstance(layer, ReLU):
        new_model.add(SquareActivation())
    else:
        new_layer = layer.__class__.from_config(layer.get_config())
        new_model.add(new_layer)
        if layer.weights:
            new_layer.set_weights(layer.get_weights())  # 自动复制权重和偏置

new_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 20)        200       
                                                                 
 average_pooling2d (AverageP  (None, 13, 13, 20)       0         
 ooling2D)                                                       
                                                                 
 batch_normalization (BatchN  (None, 13, 13, 20)       80        
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 50)        9050      
                                                                 
 square_activation (SquareAc  (None, 11, 11, 50)       0         
 tivation)                                                       
                                                      

In [ ]:
# new_model.get_weights()

第二阶段

In [9]:
#一阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow_addons as tfa

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=10,         # 如果10个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
# def lr_scheduler(epoch):
#     if epoch < 100:
#         return 0.001
#     elif epoch < 200:
#         return 0.0005
#     else:
#         return 0.0001
callbacks = [
    # tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

model.fit(
    x_train, y_train, batch_size=batch_size,
    
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True
              ,
              callbacks=callbacks
              )

score = model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

Epoch 1/4000
59/59 - 4s - loss: 0.3246 - accuracy: 0.9519 - val_loss: 0.3813 - val_accuracy: 0.9332 - 4s/epoch - 66ms/step
Epoch 2/4000
59/59 - 4s - loss: 0.3229 - accuracy: 0.9522 - val_loss: 0.3900 - val_accuracy: 0.9320 - 4s/epoch - 63ms/step
Epoch 3/4000
59/59 - 4s - loss: 0.3216 - accuracy: 0.9524 - val_loss: 0.3758 - val_accuracy: 0.9346 - 4s/epoch - 62ms/step
Epoch 4/4000
59/59 - 4s - loss: 0.3213 - accuracy: 0.9524 - val_loss: 0.4038 - val_accuracy: 0.9262 - 4s/epoch - 64ms/step
Epoch 5/4000
59/59 - 4s - loss: 0.3202 - accuracy: 0.9527 - val_loss: 0.4199 - val_accuracy: 0.9206 - 4s/epoch - 64ms/step
Epoch 6/4000
59/59 - 4s - loss: 0.3187 - accuracy: 0.9532 - val_loss: 0.4575 - val_accuracy: 0.9034 - 4s/epoch - 63ms/step
Epoch 7/4000
59/59 - 4s - loss: 0.3183 - accuracy: 0.9535 - val_loss: 0.3924 - val_accuracy: 0.9272 - 4s/epoch - 63ms/step
Epoch 8/4000
59/59 - 4s - loss: 0.3172 - accuracy: 0.9534 - val_loss: 0.3856 - val_accuracy: 0.9282 - 4s/epoch - 64ms/step
Epoch 9/4000
59/

### Serialize Model and Weights

In [10]:
model_json = new_model.to_json()
with open(os.path.join(PATH, 'model310.json'), "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
new_model.save_weights(os.path.join(PATH, 'model310.h5'))
print("Saved model to disk")

Saved model to disk


In [ ]:
# model_json = model.to_json()
# with open(os.path.join(PATH, 'model100.json'), "w") as json_file:
#     json_file.write(model_json)
# # serialize weights to HDF5
# model.save_weights(os.path.join(PATH, 'model100.h5'))
# print("Saved model to disk")

We are all done training the plain network. Next we will encrypt the network and run inference over it using FHE. Let's start with some initializations.

In [ ]:
import pyhelayers
import utils

utils.verify_memory()

print('Misc. initializations')

In [ ]:
import pyhelayers
import utils
import h5py
import os
import numpy as np
##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
# from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)



utils.verify_memory()

print('Misc. initializations')

batch_size=500

The following is a general outline of the next steps.

We encode and encrypt a neural network model, using the files that were created and saved before. An automated optimizer, that occurs during the call to encode_encrypt, will examine our network and will determine various configuration details that will allow running inference under encryption efficiently.

Next, we will demonstrate how we can encrypt data, run inference on our encrypted network, and compare the results against the expected labels.
Now let's dive in . . .

In [ ]:
he_run_req = pyhelayers.HeRunRequirements()
he_run_req.set_he_context_options([pyhelayers.DefaultContext()])
he_run_req.optimize_for_batch_size(16)

nn = pyhelayers.NeuralNet()
nn.encode_encrypt([os.path.join(PATH, "model.json"), os.path.join(PATH, "model.h5")], he_run_req)

The encode_encrypt method also initialized an HeContext object containing the keys. We obtain it now from the model so we can encrypt the input data.

In [ ]:
context = nn.get_created_he_context()

We will now load real samples of the MNIST dataset to classify. We will load the samples and the corresponding true labels from HDF5 files. We will also extract the first batch of samples and labels.

In [ ]:
with h5py.File(os.path.join(PATH, "x_test.h5")) as f:
    x_test = np.array(f["x_test"])
with h5py.File(os.path.join(PATH, "y_test.h5")) as f:
    y_test = np.array(f["y_test"])
    
# plain_samples, labels = utils.extract_batch(x_test, y_test, batch_size, 0)

plain_samples, labels = utils.extract_batch(x_test, y_test, 100, 0)
print('Batch of size',batch_size,'loaded')

To populate the input, we need to encode and then encrypt the values of the plain input under HE.

In [ ]:
model_io_encoder = pyhelayers.ModelIoEncoder(nn)
samples = pyhelayers.EncryptedData(context)
model_io_encoder.encode_encrypt(samples, [plain_samples])
print('Test data encrypted')

We now go ahead with the inference itself. We run the encrypted input through the encrypted NN to obtain encrypted results. This computation does not use the secret key and acts on completely encrypted values. Running the inference is done using the "predict" method of the NN, that receives the destination 3D structure to put the result of the computation in, and the input for the inference.

In [ ]:
utils.start_timer()

predictions = pyhelayers.EncryptedData(context)
nn.predict(predictions, samples)

duration=utils.end_timer('predict')
utils.report_duration('predict per sample',duration/batch_size)

In order to assess the results of the computation, we first need to decrypt them. This is done by an IO processor that has the secret key and is capable of decrypting the ciphertext and decoding it into plaintext version of the HE computation result.

In [ ]:
plain_predictions = model_io_encoder.decrypt_decode_output(predictions)
print('predictions',plain_predictions)

Now we compare the results against the expected labels and compute the confusion matrix and the accuracy.

In [ ]:
utils.assess_results(labels, plain_predictions)

<br>

References:

<sub><sup> 1.	LeCun, Yann and Cortes, Corinna. "MNIST handwritten digit database." (2010): </sup></sub>

<sub><sup> 2.	Gilad-Bachrach, R., Dowlin, N., Laine, K., Lauter, K., Naehrig, M. &amp; Wernsing, J.. (2016). CryptoNets: Applying Neural Networks to Encrypted Data with High Throughput and Accuracy. Proceedings of The 33rd International Conference on Machine Learning, in Proceedings of Machine Learning Research 48:201-210 Available from https://proceedings.mlr.press/v48/gilad-bachrach16.html.
</sup></sub>
